In [1]:
print(123)

123


In [4]:
from pathlib import Path

import pandas as pd

csv_path = Path("data/knowledge_base.csv")
df = pd.read_csv(csv_path)

print(df.shape)
df.head()

(10, 7)


,id,category,title,content,coach_name,skill_level,location
0,policy-001,cancellation,Private lesson cancellation,Private lessons must be cancelled at least 24 ...,NaN,NaN,Main Club
1,policy-002,booking,Court booking duration,Court bookings are available in 60-minute time...,NaN,NaN,Main Club
2,dropin-001,dropin,Beginner drop-in,The beginner drop-in is suitable for new and r...,NaN,beginner,Main Club
3,coach-001,coach,Coach Amy profile,Coach Amy specializes in beginner fundamentals...,Coach Amy,beginner,Main Club
4,coach-002,coach,Coach David profile,Coach David specializes in intermediate and ad...,Coach David,advanced,Main Club


In [5]:
documents = df.fillna("").to_dict(orient="records")

documents[:2]

[{'id': 'policy-001',
  'category': 'cancellation',
  'title': 'Private lesson cancellation',
  'content': 'Private lessons must be cancelled at least 24 hours before the scheduled start time. Late cancellations may not be eligible for a refund.',
  'coach_name': '',
  'skill_level': '',
  'location': 'Main Club'},
 {'id': 'policy-002',
  'category': 'booking',
  'title': 'Court booking duration',
  'content': 'Court bookings are available in 60-minute time slots. Customers should arrive at least 10 minutes before the booking.',
  'coach_name': '',
  'skill_level': '',
  'location': 'Main Club'}]

In [6]:
from minsearch import Index

index = Index(
    text_fields=[
        "category",
        "title",
        "content",
        "coach_name",
        "skill_level",
        "location",
    ],
    keyword_fields=["id"],
)

index.fit(documents)

In [7]:
def search(query: str, num_results: int = 5) -> list[dict]:
    results = index.search(
        query=query,
        filter_dict={},
        boost_dict={
            "title": 2.0,
            "content": 1.0,
            "coach_name": 1.5,
            "category": 1.2,
        },
        num_results=num_results,
    )

    return results

In [8]:
results = search(
    "How many hours before a private lesson do I need to cancel?"
)

for result in results:
    print(result["id"], result["title"])

course-001 Private lesson
course-002 Semi-private lesson
policy-001 Private lesson cancellation
faq-002 What to bring
coach-001 Coach Amy profile


In [9]:
results = search(
    "Which coach is suitable for a beginner learning footwork?"
)

for result in results:
    print(result["id"], result["title"])

coach-001 Coach Amy profile
coach-002 Coach David profile
dropin-001 Beginner drop-in
faq-002 What to bring
course-002 Semi-private lesson


In [10]:
prompt_template = """
You are CourtMate, an assistant for a badminton club.

Answer the QUESTION using only the information in the CONTEXT.

Rules:
- Do not invent schedules, prices, policies, or coach information.
- If the answer is not present in the context, say that the information is not available.
- Keep the answer clear and concise.

QUESTION:
{question}

CONTEXT:
{context}
""".strip()

In [11]:
entry_template = """
ID: {id}
Category: {category}
Title: {title}
Content: {content}
Coach: {coach_name}
Skill level: {skill_level}
Location: {location}
""".strip()

In [12]:
def build_prompt(
    question: str,
    search_results: list[dict],
) -> str:
    context_entries = []

    for document in search_results:
        entry = entry_template.format(**document)
        context_entries.append(entry)

    context = "\n\n".join(context_entries)

    return prompt_template.format(
        question=question,
        context=context,
    )

In [13]:
question = "How early should I cancel a private lesson?"

search_results = search(question)
prompt = build_prompt(question, search_results)

print(prompt)

You are CourtMate, an assistant for a badminton club.

Answer the QUESTION using only the information in the CONTEXT.

Rules:
- Do not invent schedules, prices, policies, or coach information.
- If the answer is not present in the context, say that the information is not available.
- Keep the answer clear and concise.

QUESTION:
How early should I cancel a private lesson?

CONTEXT:
ID: course-001
Category: course
Title: Private lesson
Content: A private lesson is a one-on-one coaching session customized to the player's level and goals.
Coach: 
Skill level: all
Location: Main Club

ID: course-002
Category: course
Title: Semi-private lesson
Content: A semi-private lesson is designed for two to four players of a similar skill level.
Coach: 
Skill level: all
Location: Main Club

ID: policy-001
Category: cancellation
Title: Private lesson cancellation
Content: Private lessons must be cancelled at least 24 hours before the scheduled start time. Late cancellations may not be eligible for a re

In [14]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise RuntimeError("OPENAI_API_KEY is missing.")

openai_client = OpenAI(api_key=api_key)

In [18]:
def rag(question: str) -> dict:
    search_results = search(question)
    prompt = build_prompt(question, search_results)
    answer = llm(prompt)

    return {
        "answer": answer,
        "sources": [
            {
                "id": result["id"],
                "title": result["title"],
            }
            for result in search_results
        ],
    }

In [16]:
def llm(prompt: str) -> str:
    model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

    response = openai_client.responses.create(
        model=model,
        input=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    return response.output_text

In [19]:
result = rag(
    "How early should I cancel a private lesson?"
)

print(result["answer"])
print(result["sources"])

You should cancel a private lesson at least 24 hours before the scheduled start time.
[{'id': 'course-001', 'title': 'Private lesson'}, {'id': 'course-002', 'title': 'Semi-private lesson'}, {'id': 'policy-001', 'title': 'Private lesson cancellation'}, {'id': 'coach-001', 'title': 'Coach Amy profile'}, {'id': 'faq-002', 'title': 'What to bring'}]
